Copyright Matlantis Corp. as contributors to Matlantis contrib project

# Cu(111)/水界面のNPzT平衡化MD

モデリングで作成したCu(111)/水界面構造を、PFPポテンシャルのもとでNPzTアンサンブル（z方向のみ圧力制御）のMDシミュレーションにより平衡化します。

このNotebookで実施する主な工程は以下のとおりです:
1. **構造の読み込み:** Step 01で作成した界面構造を読み込み、Cuの下層を固定。
2. **NPzT MDの実行:** 375 Kで100 ps（100,000ステップ）のMDシミュレーションを実施。
3. **結果の保存:** 平衡化された構造をファイルに保存。

## Step 1: ライブラリのインポートとPFPの設定

MDシミュレーションに必要なASEモジュールとPFP Calculatorをインポートし、計算条件を設定します。

In [ ]:
from pathlib import Path
import numpy as np
from time import perf_counter

# ASE
from ase.io import read, write
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.npt import NPT
from ase.md import MDLogger
from ase.constraints import FixAtoms
from ase import units

# PFP calculator
from pfcc_extras.visualize.view import view_ngl
from pfp_api_client.pfp.estimator import Estimator, EstimatorCalcMode, EstimatorMethodType
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator

estimator = Estimator(
    model_version="v8.0.0",
    method_type=EstimatorMethodType.PFVM_D3_PFVM,
    calc_mode=EstimatorCalcMode.PBE_PLUS_D3,
)
calculator = ASECalculator(estimator)

out_dir = Path('output/02_md_equilibrium')
out_dir.mkdir(exist_ok=True, parents=True)

## Step 2: 構造の読み込み

前の工程で作成・最適化した界面構造 (`cu_water_interface_opt.cif`) を読み込みます。

**note**: `output/01_modeling/` にファイルがない場合は、`assets/01_modeling/` から読み込むことで、このノートブックを独立して実行できます。

In [ ]:
import os

# outputから読み込み、なければassetsから読み込む
inp_file = "./output/01_modeling/cu_water_interface_opt.cif"
if not os.path.exists(inp_file):
    inp_file = "./assets/01_modeling/cu_water_interface_opt.cif"
    print(f"outputにファイルが見つからないため、assetsから読み込みます: {inp_file}")

atoms = read(inp_file)
atoms.calc = calculator

view_ngl(atoms, representations=["ball+stick"], w=400, h=300)

## Step 3: Cuスラブの下層を固定

Cuスラブの最下部2層（z < 4.0 Å）の原子を固定します。これにより、スラブの底面が基準位置として固定され、界面の物理的な振る舞いをより正確に再現できます。

In [ ]:
# Cuの1, 2層目を固定する

z_fix_threshold = 4.0
constraint = FixAtoms(mask=atoms.positions[:, 2] < z_fix_threshold)
atoms.set_constraint(constraint)
constraint

## Step 4: NPzT MDシミュレーションの実行

NPzT（z方向のみ圧力制御）アンサンブルでMDシミュレーションを実行します。

| パラメータ | 値 | 説明 |
|:---|:---|:---|
| アンサンブル | NPzT | z方向のみセルサイズを変動 (`mask=[0,0,1]`) |
| 温度 | 375 K | |
| 圧力 | 1.0 bar | |
| 時間刻み | 1.0 fs | |
| ステップ数 | 100,000 (= 100 ps) | |
| ログ間隔 | 1,000 ステップ | |
| `ttime` | 20.0 fs | Thermostat time constant |
| `pfactor` | 2×10⁵ GPa·fs² | Barostat parameter |

In [ ]:
%%time

# input parameters
time_step_fs = 1.0      # fs
temperature_k = 375.0   # K
pressure_bar = 1.0      # bar
num_md_steps = 100_000  # 100ps
num_interval = 1000

ttime_fs = 20.0     # Time constant [fs]
pfactor_gpa = 2e5   # Barostat parameter [GPa]

output_stem = out_dir / 'mdtraj'
log_filename = str(output_stem) + '.log'
traj_filename = str(output_stem) + '.traj'
xyz_filename = str(output_stem) + '.xyz'
cell_log_filename = str(output_stem) + '_cell.log'
print('log_filename  =', log_filename)
print('traj_filename =', traj_filename)
print('xyz_filename  =', xyz_filename)
print('cell_log_filename =', cell_log_filename)

# set the momenta corresponding to the target temperature
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_k, force_temp=True)
Stationary(atoms)

mask_matrix = np.diag([0, 0, 1])  # only z-axis cell scaling

dyn = NPT(atoms,
          time_step_fs * units.fs,
          temperature_K=temperature_k,
          externalstress=pressure_bar * units.bar,
          ttime=ttime_fs * units.fs,
          mask=mask_matrix,
          pfactor=pfactor_gpa * units.GPa * (units.fs**2),
          logfile=log_filename,
          trajectory=traj_filename,
          loginterval=num_interval)

# Print statements
def print_dyn():
    imd = dyn.get_number_of_steps()
    etot = atoms.get_total_energy()
    temp_K = atoms.get_temperature()
    volume = atoms.get_volume()
    stress = atoms.get_stress(include_ideal_gas=True) / units.GPa
    stress_ave = stress[:3].mean()
    elapsed_time = perf_counter() - start_time
    print(f"  {imd: >3}   {etot:.3f}    {temp_K:.2f}  {volume:.2f}  {stress_ave:.2f}  {stress[0]:.2f}  {stress[1]:.2f}  {stress[2]:.2f}  {stress[3]:.2f}  {stress[4]:.2f}  {stress[5]:.2f}    {elapsed_time:.3f}")

# ログファイルを開き、ヘッダーを書き込む
log_file = open(cell_log_filename, 'w')
log_file.write(f"{'Time[ps]':>12s} {'Lx[A]':>12s} {'Ly[A]':>12s} {'Lz[A]':>12s} {'Volume[A^3]':>15s}\n")

# セルの情報を記録するカスタム関数
def log_cell_info(a=atoms):
    """シミュレーションセルの寸法と体積をファイルに記録する"""

    # Cell information
    cell = a.get_cell()
    lx = cell[0, 0]
    ly = cell[1, 1]
    lz = cell[2, 2]
    
    # 体積を取得
    volume = a.get_volume()
    
    # 時間を取得 (単位をpsに変換)
    time_ps = dyn.get_time() / (1000 * units.fs)
    
    # フォーマットしてファイルに書き込む
    log_file.write(f"{time_ps:12.4f} {lx:12.6f} {ly:12.6f} {lz:12.6f} {volume:15.6f}\n")
    
    # バッファをフラッシュして、書き込みを即座に反映させる
    log_file.flush()    

dyn.attach(print_dyn, interval=num_interval)
dyn.attach(log_cell_info, interval=num_interval)
dyn.attach(MDLogger(dyn, atoms, log_filename, header=True, stress=True, peratom=True, mode="w"), interval=num_interval)

# Simulation
try:
    start_time = perf_counter()
    print(f"    imd     Etot(eV)    T(K)    volume   stress(mean,xx,yy,zz,yz,xz,xy)(GPa)  elapsed_time(sec)")
    dyn.run(num_md_steps)
finally:
    log_file.close()

## Step 5: 平衡化構造の保存

MDシミュレーション後の構造をextxyz形式で保存します。この構造は次のSteered MDの初期構造として使用されます。

In [ ]:
write(str(out_dir / 'mdtraj_eq.xyz'), atoms, format='extxyz')

## Next Step
これで、Cu(111)/水界面のNPzT平衡化MDが完了しました。
次の [03_steered_md_ja.ipynb](./03_steered_md_ja.ipynb) では、PLUMEDのMOVINGRESTRAINTを使って、Cu表面原子をz方向に引き離すSteered MDを実行します。